# Notebook 01 — Data Preparation

**Project:** Iqraa AI — Quranic recitation correction for Riwayat Qaloon an Nafi  
**Purpose:** Prepare raw audio data so it is ready for model training.

This notebook takes surah-level MP3 recordings of Sheikh Ali al-Hudhaifi
reciting in Riwayat Qaloon and produces two outputs:

1. A folder of **16 kHz mono WAV files** (one per surah) properly formatted
   for `facebook/wav2vec2-large-xlsr-53-arabic`
2. A **pairing manifest JSON** that links each WAV file to its canonical
   Qaloon ayahs from `qalon_canonical.json`

This is Step 1 of a five-notebook pipeline. No model training happens here —
only data conversion and organisation.

**Safe to re-run:** already-converted WAV files are skipped automatically.

## Cell 2 — Mount Google Drive and Configure All Paths

We mount Google Drive so the notebook can read audio files and write results
back to persistent storage. All paths are defined here in one place — if your
Drive folder layout differs, this is the only cell you need to change.

**`DRIVE_BASE`** is the root of our dataset folder on Drive.  
**`AUDIO_DIR`** is where the 114 Hudhaifi Qaloon MP3 files live.  
**`TEXT_FILE`** is the canonical Qaloon reference text (6 214 ayahs).  
**`OUTPUT_DIR`** is where this notebook writes all its outputs.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ── Change DRIVE_BASE if your Drive layout differs ────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/IqraaAI_Dataset'
AUDIO_DIR  = DRIVE_BASE + '/audio/hudhaifi_qaloon'
TEXT_FILE  = DRIVE_BASE + '/text/qalon_canonical.json'
OUTPUT_DIR = DRIVE_BASE + '/processed'
# ─────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(OUTPUT_DIR + '/wav', exist_ok=True)

print('Drive mounted.')
print(f'Audio dir  : {AUDIO_DIR}')
print(f'Text file  : {TEXT_FILE}')
print(f'Output dir : {OUTPUT_DIR}')

Mounted at /content/drive
Drive mounted.
Audio dir  : /content/drive/MyDrive/IqraaAI_Dataset/audio/hudhaifi_qaloon
Text file  : /content/drive/MyDrive/IqraaAI_Dataset/text/qalon_canonical.json
Output dir : /content/drive/MyDrive/IqraaAI_Dataset/processed


## Cell 3 — What We Are Installing and Why

We install four lightweight audio-processing libraries. We do **not** install
PyTorch or Transformers here — those are only needed in Notebook 04 (training)
and are heavy downloads. Keeping installs minimal makes this cell fast.

| Library | Purpose |
|---|---|
| **librosa** | Reads MP3/WAV, resamples audio, provides audio analysis utilities. Industry standard for audio ML preprocessing. |
| **soundfile** | Writes WAV files efficiently and reliably. librosa reads; soundfile writes. |
| **pyloudnorm** | Measures and adjusts integrated loudness (LUFS). Ensures consistent amplitude across all 114 surah recordings. |
| **tqdm** | Progress bars so we can monitor the conversion across all surahs. |

**Note:** librosa depends on `numba` and `llvmlite` which Colab pre-installs,
so the install is faster than it looks.

In [2]:
!pip install -q librosa soundfile pyloudnorm tqdm
print('Dependencies installed.')

Dependencies installed.


## Cell 5 — Our Audio Files: Surah-Level MP3s

We have **114 MP3 files** — one file per chapter (surah) of the Quran —
recorded by Sheikh Ali al-Hudhaifi in Riwayat Qaloon an Nafi.
Files are expected to be named `001.mp3` through `114.mp3`.

### Why MP3 is a problem for Wav2Vec2 training

MP3 is a compressed format designed for music streaming, not machine learning.
`facebook/wav2vec2-large-xlsr-53-arabic` was pre-trained on **16 kHz mono WAV**
audio. Feeding it raw MP3 causes two problems:

**1. Wrong sample rate.** MP3 files are typically 44 100 Hz or 48 000 Hz.
Wav2Vec2 expects exactly 16 000 Hz. A mismatch means the model processes the
wrong number of frames per second, and its acoustic representations break down.

**2. Stereo vs mono.** MP3 recordings often have two channels (left and right).
The model expects a single channel. We mix down to mono during conversion.

### What 128 kbps means

128 kbps is the bit rate — how much data is stored per second of audio.
For a single speaking voice (Quranic recitation), 128 kbps preserves all
phonetic detail including fine diacritical distinctions. It introduces no
audible artefacts that would affect training quality.

### The conversion plan

For each `001.mp3` → `114.mp3` we will:
1. Decode the MP3 and resample to **16 000 Hz**
2. Mix down to **mono** (1 channel)
3. Normalise loudness to **−23 LUFS**
4. Save as `001.wav` → `114.wav` in `OUTPUT_DIR/wav/`

In [3]:
import os

if os.path.isdir(AUDIO_DIR):
    mp3_files = sorted(
        f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.mp3')
    )
else:
    mp3_files = []

print(f'MP3 files found : {len(mp3_files)}')
print(f'First 5         : {mp3_files[:5]}')

# Total size on disk
if mp3_files:
    total_mb = sum(
        os.path.getsize(os.path.join(AUDIO_DIR, f)) for f in mp3_files
    ) / (1024 ** 2)
    print(f'Total size      : {total_mb:.1f} MB')

# Duration of the first file as a sanity check
if mp3_files:
    import librosa
    sample_path = os.path.join(AUDIO_DIR, mp3_files[0])
    y, sr = librosa.load(sample_path, sr=None, mono=True)
    duration = len(y) / sr
    print(f'\nSample file     : {mp3_files[0]}')
    print(f'Native SR       : {sr} Hz')
    print(f'Duration        : {duration/60:.1f} min  ({duration:.0f} s)')
else:
    print('\nNo MP3 files found.')
    print(f'Expected location: {AUDIO_DIR}')
    print('Upload the Hudhaifi Qaloon MP3s to that folder, then re-run.')

MP3 files found : 114
First 5         : ['001.mp3', '002.mp3', '003.mp3', '004.mp3', '005.mp3']
Total size      : 1471.3 MB

Sample file     : 001.mp3
Native SR       : 44100 Hz
Duration        : 1.0 min  (59 s)


## Cell 7 — 16 kHz Mono and LUFS Normalisation Explained

### What is a sample rate?

Audio is stored as a sequence of numbers — snapshots of the sound wave taken
at regular intervals. The **sample rate** is how many snapshots per second.
16 kHz = 16 000 snapshots per second.

Human speech contains frequencies up to roughly 8 kHz (the Nyquist limit for
16 kHz sampling). This is sufficient to capture every phonetic distinction in
Arabic speech, including the fine diacritical differences that separate Qaloon
from other riwayat. Wav2Vec2 was pre-trained at exactly 16 kHz — feeding it
audio at any other rate will silently produce wrong acoustic features.

### What is mono?

Stereo audio has two channels (left/right). The model only accepts one channel.
We average the two channels during loading — this preserves all phonetic
information while halving the data size.

### What is LUFS normalisation?

**LUFS** (Loudness Units relative to Full Scale) is a perceptual loudness
measure used in broadcast audio (EBU R128 standard). −23 LUFS is the standard
European broadcast target for speech.

Different surah recordings may have been recorded at different studios with
different microphone gain settings, resulting in different loudness levels.
Without normalisation:

- A **quiet** surah → low-amplitude input → different CNN activation patterns
  than the model saw during pre-training
- A **loud** surah → risk of clipping (values outside [−1, 1]) → distorted
  acoustic features

By normalising all files to −23 LUFS, the model sees consistent amplitude
across all 114 surahs regardless of how they were originally recorded.

In [ ]:
import librosa
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
from tqdm import tqdm

TARGET_SR   = 16_000    # Hz — required by wav2vec2-large-xlsr-53-arabic
TARGET_LUFS = -23.0     # EBU R128 broadcast loudness target
WAV_DIR     = OUTPUT_DIR + '/wav'

# Create the loudness meter once and reuse it for all files
meter = pyln.Meter(TARGET_SR)

# Track every WAV path we produce (or find already converted)
# This list is used in Cell 14 to write the completion summary.
wav_files_created = []

for mp3_name in tqdm(mp3_files, desc='Converting MP3 → WAV'):
    stem     = os.path.splitext(mp3_name)[0]   # e.g. '001'
    wav_path = os.path.join(WAV_DIR, stem + '.wav')

    # Skip files already converted — makes the cell safe to re-run
    if os.path.exists(wav_path):
        wav_files_created.append(wav_path)
        continue

    try:
        # librosa.load decodes MP3, resamples to TARGET_SR, and mixes to mono
        audio, _ = librosa.load(
            os.path.join(AUDIO_DIR, mp3_name), sr=TARGET_SR, mono=True
        )

        # Measure integrated loudness and normalise to target
        loudness = meter.integrated_loudness(audio)
        audio    = pyln.normalize.loudness(audio, loudness, TARGET_LUFS)

        # Hard-clip to [-1, 1] as a safety measure against overshoot
        audio = np.clip(audio, -1.0, 1.0)

        sf.write(wav_path, audio, TARGET_SR, subtype='PCM_16')
        wav_files_created.append(wav_path)

    except Exception as e:
        print(f'  ERROR {mp3_name}: {e}')

total_wav_mb = (
    sum(os.path.getsize(p) for p in wav_files_created) / (1024 ** 2)
    if wav_files_created else 0
)
print(f'\nConverted      : {len(wav_files_created)} files')
print(f'Total WAV size : {total_wav_mb:.1f} MB')
print(f'Saved to       : {WAV_DIR}')

Converting MP3 → WAV:   1%|          | 1/114 [00:00<00:26,  4.21it/s]

## Cell 9 — What qalon_canonical.json Contains

The file `qalon_canonical.json` is our **ground truth** — the written form of
what a perfect Qaloon recitation sounds like.

### Schema

It is a JSON object (Python dict) with **6 214 entries**. Each key is a
6-character string encoding the surah and ayah number:

```
001001  →  Surah 1,   Ayah 1   (first ayah of Al-Fatihah)
001007  →  Surah 1,   Ayah 7   (last ayah of Al-Fatihah)
002001  →  Surah 2,   Ayah 1   (first ayah of Al-Baqarah)
114006  →  Surah 114, Ayah 6   (last ayah of An-Nas)
```

Each value is the **fully diacritized** Arabic text of that ayah in the
KFGQPC (King Fahd Glorious Quran Printing Complex) edition for Qaloon.
Diacritics (tashkeel) are preserved in full — this is critical because the
whole purpose of the system is to detect diacritical errors.

**Note on trailing characters:** each value ends with a non-breaking space
and an Arabic-Indic numeral (the ayah number in Arabic script). Notebook 02
will strip these during text normalisation.

### How it pairs with the audio

Our MP3/WAV files are named `001.wav` to `114.wav` — one file per surah.
The key prefix tells us the surah: keys starting with `001` belong to surah 1,
which pairs with `001.wav`. This notebook builds a manifest that makes that
mapping explicit.

In [ ]:
import json
from collections import Counter

with open(TEXT_FILE, encoding='utf-8') as f:
    quran = json.load(f)

print(f'Total ayahs: {len(quran)}')
print()

# Show first 3 entries (key + surah/ayah parse + text)
print('First 3 entries:')
for key in list(quran.keys())[:3]:
    surah_num = int(key[:3])
    ayah_num  = int(key[3:])
    print(f'  Key: {key}  →  Surah {surah_num}, Ayah {ayah_num}')
    print(f'  Text: {quran[key]}')
    print()

# Count ayahs per surah for surahs 1–5
surah_counts = Counter(int(k[:3]) for k in quran.keys())
print('Ayahs per surah (1–5):')
for s in range(1, 6):
    print(f'  Surah {s}: {surah_counts[s]} ayahs')

# Verify Al-Fatihah has exactly 7 ayahs
fatihah_keys = [k for k in quran.keys() if k.startswith('001')]
print(f'\nAl-Fatihah keys : {fatihah_keys}')
assert len(fatihah_keys) == 7, f'Expected 7 ayahs, got {len(fatihah_keys)}'
print('Confirmed: Al-Fatihah has 7 ayahs.')

## Cell 11 — The Pairing Logic

Each WAV file covers an entire surah. Our goal is to know, for each WAV file,
exactly which ayahs it contains and what their diacritized texts are.

### The mapping

The logic is straightforward because the naming convention is consistent:

- `001.wav` → surah 1 → all JSON keys where the first 3 characters are `001`
- `002.wav` → surah 2 → all JSON keys where the first 3 characters are `002`
- ... and so on to `114.wav`

### What the manifest is

We build a **pairing manifest** — a JSON list where each entry describes one
surah: which WAV file contains it, and the full ordered list of its ayahs with
their keys and texts.

```json
[
  {
    "surah": 1,
    "audio_path": ".../processed/wav/001.wav",
    "ayahs": [
      {"key": "001001", "ayah_num": 1, "text": "..."},
      ...
    ]
  },
  ...
]
```

### What the manifest does NOT do

It does **not** tell us where in the WAV file each individual ayah starts and
ends — that is the job of Notebook 03 (alignment), which will use forced
alignment or energy-based segmentation to find precise boundaries.

The manifest also lets us immediately spot any missing audio or text before
investing hours in training on incomplete data.

In [ ]:
import json
import os
from collections import defaultdict

WAV_DIR = OUTPUT_DIR + '/wav'

# Collect available WAV stems (e.g. {'001', '002', ...})
if os.path.isdir(WAV_DIR):
    available_wav_stems = {
        os.path.splitext(f)[0]
        for f in os.listdir(WAV_DIR)
        if f.lower().endswith('.wav')
    }
else:
    available_wav_stems = set()

# Group JSON keys by surah number, keeping ayahs in order
surah_ayahs = defaultdict(list)
for key, text in quran.items():
    surah_num = int(key[:3])
    ayah_num  = int(key[3:])
    surah_ayahs[surah_num].append({
        'key'     : key,
        'ayah_num': ayah_num,
        'text'    : text,
    })
for s in surah_ayahs:
    surah_ayahs[s].sort(key=lambda x: x['ayah_num'])

# Compute overlap / gaps
text_surahs  = set(surah_ayahs.keys())
audio_surahs = {int(stem) for stem in available_wav_stems}
paired_surahs  = text_surahs & audio_surahs
missing_audio  = text_surahs - audio_surahs
missing_text   = audio_surahs - text_surahs

# Build the manifest for paired surahs
manifest = []
for surah_num in sorted(paired_surahs):
    stem     = f'{surah_num:03d}'
    wav_path = os.path.join(WAV_DIR, stem + '.wav')
    manifest.append({
        'surah'     : surah_num,
        'audio_path': wav_path,
        'ayahs'     : surah_ayahs[surah_num],
    })

manifest_path = OUTPUT_DIR + '/pairing_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f'Surahs paired        : {len(paired_surahs)}')
if missing_audio:
    print(f'Surahs missing audio : {len(missing_audio)} '
          f'(first 10: {sorted(missing_audio)[:10]})')
else:
    print('Surahs missing audio : 0  ✓')
if missing_text:
    print(f'Surahs missing text  : {len(missing_text)} '
          f'(first 10: {sorted(missing_text)[:10]})')
else:
    print('Surahs missing text  : 0  ✓')

print(f'\nManifest saved to    : {manifest_path}')

# Preview the first entry
if manifest:
    first = manifest[0]
    print(f'\nPreview — Surah {first["surah"]}:')
    print(f'  Audio : {first["audio_path"]}')
    print(f'  Ayahs : {len(first["ayahs"])}')
    print(f'  First : {first["ayahs"][0]}')

## Cell 13 — Summary and What Comes Next

### What this notebook produced

**`OUTPUT_DIR/wav/`**  
One 16 kHz mono WAV file per surah, loudness-normalised to −23 LUFS.
These files are ready to be split into ayah-level segments by Notebook 03.

**`OUTPUT_DIR/pairing_manifest.json`**  
A JSON list linking every WAV file to its surah number and the full ordered
list of diacritized ayah texts from `qalon_canonical.json`.

**`OUTPUT_DIR/notebook_01_complete.json`**  
A completion marker written by Cell 14 with a count summary.

### What Notebook 02 will do

**Notebook 02 — Text Processing** takes the canonical text and prepares it
for use as CTC training labels:

1. Apply Qaloon-specific Arabic text normalisation — remove punctuation,
   tatweel, and the trailing ayah numerals, while keeping all diacritics
2. Build the **character-level vocabulary** — every unique character in the
   normalised corpus becomes a CTC output token
3. Save `labels.json` (segment ID → normalised text) and `vocab.json`
   (character → integer index)

The vocabulary built in Notebook 02 directly defines what characters the
trained model can predict, so it must include every diacritic in the Qaloon
corpus.

In [ ]:
import json

total_ayahs_paired = sum(len(entry['ayahs']) for entry in manifest)

summary = {
    'notebook'         : '01_data_preparation',
    'status'           : 'complete',
    'wav_files_created': len(wav_files_created),
    'surahs_paired'    : len(paired_surahs),
    'total_ayahs'      : total_ayahs_paired,
}

marker_path = OUTPUT_DIR + '/notebook_01_complete.json'
with open(marker_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Notebook 01 complete.')
print(json.dumps(summary, indent=2))
print(f'\nCompletion marker saved to: {marker_path}')